### Bronze Layer Inspection

Goal: verify every column bronze_01 / bronze_02 actually produce, cross check
the flag logic against real sample rows (not just the summary counts), and
catch row-count fan-out issues before they hit Silver.

Run this top to bottom after `run_pipeline.py bronze`.

In [1]:
import duckdb
from pathlib import Path
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', 80)

DB_PATH = Path().resolve().parent / "phantomproof.duckdb"
con = duckdb.connect(str(DB_PATH), read_only=True)
print(f"Connected read only to {DB_PATH.name}")

Connected read only to phantomproof.duckdb


#### 1. What columns actually exist?

Do this before writing any query against bronze tables

In [2]:
print("bronze.events columns:")
print(con.execute("DESCRIBE bronze.events").df().to_string())

bronze.events columns:
                    column_name               column_type null   key default extra
0                      event_id                   VARCHAR  YES  None    None  None
1                    event_type                   VARCHAR  YES  None    None  None
2               event_timestamp                 TIMESTAMP  YES  None    None  None
3                       node_id                   VARCHAR  YES  None    None  None
4                     device_id                   VARCHAR  YES  None    None  None
5                    product_id                   VARCHAR  YES  None    None  None
6                units_sold_raw                   VARCHAR  YES  None    None  None
7                    units_sold                    BIGINT  YES  None    None  None
8                     units_rto                   INTEGER  YES  None    None  None
9             units_transferred                   INTEGER  YES  None    None  None
10                stock_on_hand                   INTEGER  YES  

In [3]:
print("bronze.nodes columns:")
print(con.execute("DESCRIBE bronze.nodes").df().to_string())

bronze.nodes columns:
                 column_name               column_type null   key default extra
0                    node_id                   VARCHAR  YES  None    None  None
1                  node_type                   VARCHAR  YES  None    None  None
2                  node_name                   VARCHAR  YES  None    None  None
3                       city                   VARCHAR  YES  None    None  None
4                       zone                   VARCHAR  YES  None    None  None
5                  zone_type                   VARCHAR  YES  None    None  None
6                       tier                   VARCHAR  YES  None    None  None
7             parent_node_id                   VARCHAR  YES  None    None  None
8             capacity_units                   INTEGER  YES  None    None  None
9        capacity_cold_units                   INTEGER  YES  None    None  None
10   capacity_beverage_units                   INTEGER  YES  None    None  None
11        capacity

#### 2. Row count reconciliation

input_rows -> rejected -> bronze rows should reconcile. If bronze_rows is
*higher* than input_rows minus rejected, something is fanning out (e.g. a
join on a non-unique key).

In [4]:
counts = con.execute("""
    SELECT
        (SELECT COUNT(*) FROM raw.events_chaotic)      AS raw_input_rows,
        (SELECT COUNT(*) FROM bronze.rejected_events)  AS rejected_rows,
        (SELECT COUNT(*) FROM bronze.events)           AS bronze_rows,
        (SELECT COUNT(DISTINCT event_id) FROM bronze.events) AS distinct_event_ids
""").df()
counts["expected_bronze_rows"] = counts["raw_input_rows"] - counts["rejected_rows"]
counts["unexplained_delta"] = counts["bronze_rows"] - counts["expected_bronze_rows"]
counts.T

,0
raw_input_rows,436124
rejected_rows,0
bronze_rows,436124
distinct_event_ids,435974
expected_bronze_rows,436124
unexplained_delta,0


If `unexplained_delta` is not 0, some row is appearing more than once — most
likely a join fan-out (see the near-duplicate check in section 6).

#### 3. Flag summary (fresh count, not relying on ingestion_log)

In [5]:
flag_cols = [c for c in con.execute("DESCRIBE bronze.events").df()["column_name"] if c.startswith("flag_")]

select_parts = ", ".join(f"SUM(CAST({c} AS INTEGER)) AS {c}" for c in flag_cols)
flag_summary = con.execute(f"SELECT {select_parts} FROM bronze.events").df()
flag_summary.T.rename(columns={0: "rows_flagged"})

,rows_flagged
flag_utc_drift,85347.0
flag_future_timestamp,558.0
flag_batch_heartbeat,1354.0
flag_midnight_rollover,10946.0
flag_lag_smear,2510.0
flag_extreme_lag,4960.0
flag_schema_poison_units,50.0
flag_decimal_separator,887.0
flag_null_rto_reason,20.0
flag_uom_mismatch,0.0


#### 4. Sample rows per flag (check whether the flag logic makes sense)

This is the part that actually catches bugs. Summary counts of 0 or
suspiciously huge numbers only tell you *something* is wrong — these
samples tell you *what*.

In [6]:
print("-- flag_utc_drift sample --")
print(con.execute("""
    SELECT event_id, device_id, event_timestamp, flag_utc_drift
    FROM bronze.events
    WHERE device_id IN (SELECT device_id FROM raw.dim_scanner WHERE battery_backed_rtc = FALSE)
    LIMIT 10
""").df())

-- flag_utc_drift sample --
      event_id     device_id     event_timestamp  flag_utc_drift
0  EVT_0346284  SCAN_ANN_002 2026-05-22 23:11:00            True
1  EVT_0346287  SCAN_ANN_001 2026-05-23 11:03:00            True
2  EVT_0346288  SCAN_ANN_001 2026-05-23 18:00:00            True
3  EVT_0346289  SCAN_ANN_001 2026-05-23 01:43:00            True
4  EVT_0346290  SCAN_ANN_001 2026-05-23 15:02:00            True
5  EVT_0346291  SCAN_ANN_001 2026-05-23 11:11:00            True
6  EVT_0346292  SCAN_ANN_001 2026-05-23 12:11:00            True
7  EVT_0346293  SCAN_ANN_002 2026-05-23 17:11:00            True
8  EVT_0346296  SCAN_ANN_001 2026-05-23 04:03:00            True
9  EVT_0346297  SCAN_ANN_002 2026-05-23 05:03:00            True


In [7]:
print("-- flag_schema_poison_units: raw vs cleaned value --")
print(con.execute("""
    SELECT event_id, units_sold_raw, units_sold, flag_schema_poison_units
    FROM bronze.events
    WHERE units_sold_raw ILIKE '%units%'
    LIMIT 10
""").df())

-- flag_schema_poison_units: raw vs cleaned value --
      event_id units_sold_raw  units_sold  flag_schema_poison_units
0  EVT_0030293       11 units          11                      True
1  EVT_0029246        1 units           1                      True
2  EVT_0076788        3 units           3                      True
3  EVT_0066438        1 units           1                      True
4  EVT_0102601        1 units           1                      True
5  EVT_0288145        1 units           1                      True
6  EVT_0106845        2 units           2                      True
7  EVT_0348345        5 units           5                      True
8  EVT_0381115        5 units           5                      True
9  EVT_0405659        1 units           1                      True


In [8]:
print("-- flag_decimal_separator: raw vs cleaned value --")
print(con.execute("""
    SELECT event_id, processing_lag_hours_raw, processing_lag_hours, flag_decimal_separator
    FROM bronze.events
    WHERE processing_lag_hours_raw LIKE '%,%'
    LIMIT 10
""").df())

-- flag_decimal_separator: raw vs cleaned value --
      event_id processing_lag_hours_raw  processing_lag_hours  flag_decimal_separator
0  EVT_0030608                    0,475                 0.475                    True
1  EVT_0029128                     0,72                 0.720                    True
2  EVT_0029519                    0,034                 0.034                    True
3  EVT_0029590                    1,125                 1.125                    True
4  EVT_0030171                    0,346                 0.346                    True
5  EVT_0030265                    1,053                 1.053                    True
6  EVT_0030420                    0,047                 0.047                    True
7  EVT_0076094                    1,572                 1.572                    True
8  EVT_0076097                    1,364                 1.364                    True
9  EVT_0076285                    0,256                 0.256                    True


In [9]:
print("-- flag_gst_drift: check whether xml_products set is even populated --")
print(con.execute("""
    SELECT s.edi_format, COUNT(*) AS product_count
    FROM raw.dim_product p
    JOIN raw.dim_supplier s ON p.supplier_id = s.supplier_id
    GROUP BY s.edi_format
""").df())

-- flag_gst_drift: check whether xml_products set is even populated --
     edi_format  product_count
0      SFTP_XML             21
1           API             16
2      SFTP_CSV             35
3  email_manual             28


In [10]:
print("-- flag_uom_mismatch sample: is this actually catching cases-vs-units, or every small transfer? --")
print(con.execute("""
    SELECT event_id, product_id, units_transferred, flag_uom_mismatch
    FROM bronze.events
    WHERE flag_uom_mismatch = TRUE
    LIMIT 15
""").df())
print()
print("Total inbound_transfer rows on email_manual products (denominator for comparison):")
print(con.execute("""
    SELECT COUNT(*) FROM bronze.events e
    JOIN raw.dim_product p ON e.product_id = p.product_id
    JOIN raw.dim_supplier s ON p.supplier_id = s.supplier_id
    WHERE e.event_type = 'inbound_transfer' AND s.edi_format = 'email_manual'
""").df())

-- flag_uom_mismatch sample: is this actually catching cases-vs-units, or every small transfer? --
Empty DataFrame
Columns: [event_id, product_id, units_transferred, flag_uom_mismatch]
Index: []

Total inbound_transfer rows on email_manual products (denominator for comparison):
   count_star()
0         25046


#### 5. Duplicate event_id check (the fan-out suspect)

The chaos engine copies rows for exact/near duplicates *without* changing
`event_id`. If `near_dupe_detection` (or any other CTE) joins back on
`event_id` alone, every copy multiplies against every other copy.

In [11]:
print("How many event_ids appear more than once in bronze.events?")
print(con.execute("""
    SELECT COUNT(*) AS duplicated_event_ids FROM (
        SELECT event_id FROM bronze.events
        GROUP BY event_id HAVING COUNT(*) > 1
    )
""").df())
print()
print("Sample of a duplicated event_id and all its rows:")
print(con.execute("""
    WITH dupes AS (
        SELECT event_id FROM bronze.events
        GROUP BY event_id HAVING COUNT(*) > 1
        LIMIT 1
    )
    SELECT b.* FROM bronze.events b
    JOIN dupes d ON b.event_id = d.event_id
""").df())

How many event_ids appear more than once in bronze.events?
   duplicated_event_ids
0                   150

Sample of a duplicated event_id and all its rows:
      event_id            event_type         event_timestamp    node_id     device_id product_id units_sold_raw  units_sold  units_rto  units_transferred  stock_on_hand source_node destination_node processing_lag_hours_raw  processing_lag_hours rto_reason                  session_id  flag_utc_drift  flag_future_timestamp  flag_batch_heartbeat  flag_midnight_rollover  flag_lag_smear  flag_extreme_lag  flag_schema_poison_units  flag_decimal_separator  flag_null_rto_reason  flag_uom_mismatch  flag_ghost_inventory  flag_exact_duplicate  flag_near_duplicate  flag_reverse_logistics_void  flag_lateral_orphan  flag_negative_stock  flag_gst_drift  flag_sku_migration  flag_fat_finger  flag_late_arriving  flag_out_of_order                      ingested_at         source_file          row_hash  any_flag  flag_count
0  EVT_0056829  customer_fu

#### 6. bronze.nodes: city corruption + orphan checks

In [12]:
print(con.execute("""
    SELECT node_type, COUNT(*) AS node_count,
           SUM(CAST(any_flag AS INTEGER)) AS flagged,
           SUM(CAST(flag_city_corrupt AS INTEGER)) AS city_corrupt,
           SUM(CAST(flag_orphaned_node AS INTEGER)) AS orphaned,
           SUM(CAST(flag_out_of_india AS INTEGER)) AS out_of_india
    FROM bronze.nodes
    GROUP BY node_type ORDER BY node_type
""").df())

      node_type  node_count  flagged  city_corrupt  orphaned  out_of_india
0     darkstore         100      2.0           2.0       0.0           0.0
1    mother_hub           8      1.0           1.0       0.0           0.0
2  regional_hub          12      0.0           0.0       0.0           0.0


In [13]:
print(con.execute("""
    SELECT node_id, city AS city_chaotic, city_clean, node_type
    FROM bronze.nodes WHERE flag_city_corrupt = TRUE
""").df())

     node_id    city_chaotic city_clean   node_type
0  MH_KOL_01  Sector 12 || 🏠    Kolkata  mother_hub
1  DS_IND_02    AREA-999-###  Bangalore   darkstore
2  DS_GMN_01    AREA-999-###    Lucknow   darkstore


#### 7. Rejected rows check

In [14]:
print(con.execute("""
    SELECT rejection_reason, COUNT(*) FROM bronze.rejected_events
    GROUP BY rejection_reason
""").df())

Empty DataFrame
Columns: [rejection_reason, count_star()]
Index: []


close the connection once inspection done

In [15]:
def new_func():
    con.close()
    print("Connection closed cleanly.")

new_func()

Connection closed cleanly.
